# DBF Comparator

Анализ и сравнение структуры и заполненности двух DBF-файлов:
- **`reference.dbf`** — эталонный файл
- **`test.dbf`** — результат обработки NiFi / junior-парсера

---

## 0. Импорт библиотек и конфигурация

In [ ]:
import pandas as pd
import numpy as np
from dbfread import DBF
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

# ─────────────────────────────────────────────
# НАСТРОЙКИ: укажите пути к файлам и ключевые поля
# ─────────────────────────────────────────────
REFERENCE_PATH = 'reference.dbf'   # путь к эталонному DBF
TEST_PATH      = 'test.dbf'        # путь к проверяемому DBF

# Поле(а) для JOIN-анализа (список строк или None для отключения)
# Пример: JOIN_KEYS = ['ID', 'CODE']
JOIN_KEYS = None

# Порог для сигнализации подозрительно низкой заполненности (в %)
FILL_ALERT_THRESHOLD = 20.0  # столбцы с заполненностью < этого % помечаются

# Порог значимого отклонения заполненности между файлами (в %)
FILL_DELTA_THRESHOLD = 10.0

# Считать ли нулевые значения (0, '0', '0.0' и т.д.) пустыми?
# True  — нули трактуются как отсутствие данных (типично для DBF)
# False — нули считаются валидными значениями
TREAT_ZEROS_AS_EMPTY = True

print('✓ Конфигурация загружена')

## 1. Загрузка DBF-файлов

In [ ]:
def load_dbf(path: str) -> pd.DataFrame:
    """
    Читает DBF-файл и возвращает DataFrame.
    Нормализует имена столбцов: приводит к верхнему регистру,
    убирает пробелы по краям.
    Пустые строки заменяет на np.nan для корректного подсчёта.
    """
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f'Файл не найден: {path}')

    table = DBF(path, load=True, encoding='cp866')  # cp866 — типичная кодировка русских DBF
    df = pd.DataFrame(iter(table))

    # Нормализация имён столбцов
    df.columns = [c.strip().upper() for c in df.columns]

    # Привести строковые значения: убрать пробелы, заменить '' на NaN.
    # ВАЖНО: dtype == object недостаточно — dbfread возвращает даты (datetime.date),
    # булевы (bool) и другие Python-объекты тоже с dtype=object.
    # Применяем .str только к колонкам, где все непустые значения — действительно str.
    for col in df.columns:
        if df[col].dtype == object:
            # Берём первое непустое значение и проверяем его тип
            sample = df[col].dropna()
            if sample.empty or not isinstance(sample.iloc[0], str):
                # Дата, bool или иной не-строковый Python-объект — пропускаем
                continue
            df[col] = df[col].str.strip().replace('', np.nan)

    return df


ref_df  = load_dbf(REFERENCE_PATH)
test_df = load_dbf(TEST_PATH)

print(f'reference.dbf → {ref_df.shape[0]:,} строк, {ref_df.shape[1]} столбцов')
print(f'test.dbf      → {test_df.shape[0]:,} строк, {test_df.shape[1]} столбцов')

In [ ]:
# Предпросмотр первых строк эталона
print('── reference.dbf (первые 3 строки) ──')
ref_df.head(3)

In [ ]:
# Предпросмотр первых строк тестового файла
print('── test.dbf (первые 3 строки) ──')
test_df.head(3)

## 2. Сравнение структуры (схемы колонок)

In [ ]:
ref_cols  = list(ref_df.columns)
test_cols = list(test_df.columns)

ref_set  = set(ref_cols)
test_set = set(test_cols)

common_cols     = sorted(ref_set & test_set)
only_in_ref     = sorted(ref_set - test_set)
only_in_test    = sorted(test_set - ref_set)

print('=' * 60)
print('СТРУКТУРА КОЛОНОК')
print('=' * 60)
print(f'  Эталон       ({len(ref_cols):>3} кол.): {ref_cols}')
print(f'  Проверяемый  ({len(test_cols):>3} кол.): {test_cols}')
print()
print(f'  Общих столбцов         : {len(common_cols)}')
print(f'  Только в эталоне       : {only_in_ref  if only_in_ref  else "— нет"}')
print(f'  Только в проверяемом   : {only_in_test if only_in_test else "— нет"}')

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Сравнение схемы: три независимых среза
#
# ПОЧЕМУ не используем наивное позиционное сравнение:
#   Если в test вставлен лишний столбец посередине, все последующие
#   позиции сдвигаются. Позиционный diff покажет «⚠ порядок» для
#   столбцов, которые просто сдвинулись, а не изменились.
#   Пример:  ref=[A,B,C]  test=[A,B,EXTRA,C]
#     pos 3: ref=C  test=EXTRA → «⚠ порядок»  ← ЛОЖНО
#     pos 4: ref='' test=C    → «→ только test» ← ЛОЖНО
#
# Правильный подход: множественный анализ + позиции общих столбцов.
# ─────────────────────────────────────────────────────────────────

ref_set  = set(ref_cols)
test_set = set(test_cols)

# 1. Только в эталоне
only_ref_list  = [c for c in ref_cols  if c not in test_set]
# 2. Только в тесте
only_test_list = [c for c in test_cols if c not in ref_set]
# 3. Общие
common_list    = [c for c in ref_cols  if c in test_set]

print('=' * 60)
print('А. МНОЖЕСТВЕННЫЙ АНАЛИЗ (по принадлежности)')
print('=' * 60)

rows_membership = []
for col in sorted(ref_set | test_set):
    in_ref  = col in ref_set
    in_test = col in test_set
    if in_ref and in_test:
        status = '✓ общий'
    elif in_ref:
        status = '← только в reference'
    else:
        status = '→ только в test'
    rows_membership.append({
        'столбец':    col,
        'reference':  '✓' if in_ref  else '—',
        'test':       '✓' if in_test else '—',
        'статус':     status,
    })

membership_df = pd.DataFrame(rows_membership)
display(membership_df)

print(f'\nИтого: {len(ref_cols)} в reference | {len(test_cols)} в test | '
      f'{len(common_list)} общих | '
      f'{len(only_ref_list)} только в ref | {len(only_test_list)} только в test')

# ─────────────────────────────────────────────────────────────────
print()
print('=' * 60)
print('Б. ПОЗИЦИИ ОБЩИХ СТОЛБЦОВ (есть в обоих файлах)')
print('=' * 60)

ref_pos  = {col: i+1 for i, col in enumerate(ref_cols)}
test_pos = {col: i+1 for i, col in enumerate(test_cols)}

rows_positions = []
for col in common_list:
    rp = ref_pos[col]
    tp = test_pos[col]
    order_ok = '✓' if rp == tp else f'⚠ ref={rp} test={tp}'
    rows_positions.append({
        'столбец':       col,
        'позиция ref':   rp,
        'позиция test':  tp,
        'порядок':       order_ok,
    })

positions_df = pd.DataFrame(rows_positions)
display(positions_df)

reordered = [r['столбец'] for r in rows_positions if r['порядок'] != '✓']
if reordered:
    print(f'\n⚠ Столбцы с отличающейся позицией ({len(reordered)}): {reordered}')
else:
    print('\n✓ Все общие столбцы стоят на одинаковых позициях')

## 3. Анализ заполненности столбцов

In [ ]:
# Значения, которые считаются «пустыми» несмотря на то что технически не NaN.
# Нули в числовых/строковых полях DBF часто означают отсутствие данных.
# Добавьте сюда любые другие sentinel-значения вашего источника.
ZERO_SENTINELS_STR = {'0', '0.0', '0.00', '00000000', '000'}  # строковые нули
ZERO_SENTINELS_NUM = {0, 0.0}                                   # числовые нули

def is_empty_value(val) -> bool:
    """Возвращает True если значение считается пустым/нулевым."""
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return True
    if not TREAT_ZEROS_AS_EMPTY:
        return False
    if isinstance(val, str) and val.strip() in ZERO_SENTINELS_STR:
        return True
    if isinstance(val, (int, float)) and val in ZERO_SENTINELS_NUM:
        return True
    return False


def fill_stats(df: pd.DataFrame, label: str) -> pd.DataFrame:
    """
    Считает заполненность каждого столбца DataFrame.
    «Пустыми» считаются: NaN, None, пустая строка,
    а также нулевые sentinel-значения (0, '0', '0.0' и т.д.) —
    типичные для DBF-полей, где отсутствие данных кодируется нулём.

    Возвращает DataFrame со столбцами:
      column, total, filled, empty_null, empty_zero, empty_total,
      fill_pct, empty_pct
    """
    total = len(df)
    rows = []
    for col in df.columns:
        series = df[col]

        # NaN / None
        null_mask = series.isna()

        # Нулевые sentinel-значения среди непустых
        zero_mask = series.apply(
            lambda v: (not (isinstance(v, float) and np.isnan(v)))
                      and (v is not None)
                      and is_empty_value(v)
        )

        empty_null  = int(null_mask.sum())
        empty_zero  = int(zero_mask.sum())
        empty_total = empty_null + empty_zero
        filled      = total - empty_total

        rows.append({
            'column':      col,
            'total':       total,
            'filled':      filled,
            'empty_null':  empty_null,   # NaN / None
            'empty_zero':  empty_zero,   # нулевые sentinel
            'empty_total': empty_total,  # всего пустых
            'fill_pct':    round(filled      / total * 100, 2) if total > 0 else 0.0,
            'empty_pct':   round(empty_total / total * 100, 2) if total > 0 else 0.0,
        })
    result = pd.DataFrame(rows).set_index('column')
    result.columns = pd.MultiIndex.from_tuples(
        [(label, c) for c in result.columns]
    )
    return result


ref_fill  = fill_stats(ref_df,  'reference')
test_fill = fill_stats(test_df, 'test')

print('Заполненность эталонного файла:')
ref_fill

In [ ]:
print('Заполненность тестового файла:')
test_fill

In [ ]:
# Объединённая таблица по общим столбцам
if common_cols:
    combined = ref_fill.loc[
        ref_fill.index.isin(common_cols)
    ].join(
        test_fill.loc[test_fill.index.isin(common_cols)],
        how='inner'
    )

    # Дельта заполненности (reference - test)
    combined[('delta', 'fill_pct_diff')] = (
        combined[('reference', 'fill_pct')] - combined[('test', 'fill_pct')]
    ).round(2)

    print('Сравнение заполненности по общим столбцам:')
    display(combined)
else:
    print('Нет общих столбцов для сравнения заполненности.')

## 4. JOIN-анализ (строки по ключевым полям)

In [ ]:
if JOIN_KEYS is not None:
    # Проверяем наличие ключей в обоих файлах
    missing_in_ref  = [k for k in JOIN_KEYS if k not in ref_df.columns]
    missing_in_test = [k for k in JOIN_KEYS if k not in test_df.columns]

    if missing_in_ref:
        print(f'⚠ Ключи отсутствуют в эталоне: {missing_in_ref}')
    elif missing_in_test:
        print(f'⚠ Ключи отсутствуют в тестовом файле: {missing_in_test}')
    else:
        merged = ref_df[JOIN_KEYS].merge(
            test_df[JOIN_KEYS],
            on=JOIN_KEYS,
            how='outer',
            indicator=True
        )

        only_left  = merged[merged['_merge'] == 'left_only'].drop(columns='_merge')
        only_right = merged[merged['_merge'] == 'right_only'].drop(columns='_merge')
        both       = merged[merged['_merge'] == 'both'].drop(columns='_merge')

        print(f'JOIN по ключам: {JOIN_KEYS}')
        print(f'  Совпали:                       {len(both):>7,}')
        print(f'  Только в эталоне (left only):  {len(only_left):>7,}')
        print(f'  Только в тесте  (right only):  {len(only_right):>7,}')

        if not only_left.empty:
            print('\nПервые строки только в эталоне:')
            display(only_left.head(10))

        if not only_right.empty:
            print('\nПервые строки только в тестовом файле:')
            display(only_right.head(10))
else:
    print('JOIN-анализ отключён. Укажите JOIN_KEYS в ячейке конфигурации.')

## 5. Итоговое резюме

In [ ]:
issues = []
ok     = []

# ── Количество строк ──────────────────────────────────────────────
row_diff = ref_df.shape[0] - test_df.shape[0]
if row_diff == 0:
    ok.append(f'Количество строк совпадает: {ref_df.shape[0]:,}')
else:
    sign = '+' if row_diff > 0 else ''
    issues.append(
        f'Расхождение по строкам: эталон {ref_df.shape[0]:,}, '
        f'тест {test_df.shape[0]:,} (diff {sign}{row_diff:,})'
    )

# ── Структура ─────────────────────────────────────────────────────
if only_in_ref:
    issues.append(f'Отсутствуют в test.dbf ({len(only_in_ref)} кол.): {only_in_ref}')
else:
    ok.append('Все столбцы эталона присутствуют в тестовом файле')

if only_in_test:
    issues.append(f'Лишние столбцы в test.dbf ({len(only_in_test)} кол.): {only_in_test}')
else:
    ok.append('Лишних столбцов в тестовом файле нет')

# ── Заполненность ─────────────────────────────────────────────────
if common_cols:
    # Столбцы с подозрительно низкой заполненностью в тесте
    low_fill = test_fill.loc[
        test_fill.index.isin(common_cols) &
        (test_fill[('test', 'fill_pct')] < FILL_ALERT_THRESHOLD)
    ]
    if not low_fill.empty:
        for col in low_fill.index:
            pct = low_fill.loc[col, ('test', 'fill_pct')]
            issues.append(f'Подозрительно низкая заполненность в test: {col} = {pct:.1f}%')

    # Значительная дельта между эталоном и тестом
    if 'combined' in dir() and not combined.empty:
        big_delta = combined[
            combined[('delta', 'fill_pct_diff')].abs() > FILL_DELTA_THRESHOLD
        ]
        for col in big_delta.index:
            ref_pct  = big_delta.loc[col, ('reference', 'fill_pct')]
            test_pct = big_delta.loc[col, ('test',      'fill_pct')]
            delta    = big_delta.loc[col, ('delta',     'fill_pct_diff')]
            issues.append(
                f'Значимая разница заполненности — {col}: '
                f'эталон {ref_pct:.1f}%, тест {test_pct:.1f}% (Δ {delta:+.1f}%)'
            )

# ── Вывод резюме ──────────────────────────────────────────────────
print('=' * 65)
print('ИТОГОВОЕ РЕЗЮМЕ')
print('=' * 65)

if ok:
    print('\n✓ Всё в порядке:')
    for msg in ok:
        print(f'   ✓  {msg}')

if issues:
    print(f'\n⚠  Обнаружено проблем: {len(issues)}')
    for i, msg in enumerate(issues, 1):
        print(f'   {i:02d}. {msg}')
else:
    print('\n✓ Серьёзных расхождений не обнаружено.')

print('\n' + '=' * 65)